## Indicator Calculator
### - Delta, Cumulative Delta

### - Simple Moving Averages:
Formula:
$$
MA_t = \frac{1}{n} \sum_{i=0}^{n-1} x_{t-i}
$$

- MA*t* = Moving Average of Time *t*
- N = Window Period Number
- X*t*-*i* = Series of Time Value *t* - *i* 

Python Code:

```python
    ma =  data.rolling(window=window).mean()
```
### - Weighted Moving Averages:
Formula:
$$
WMA_t = \frac{\sum_{i=1}^{n} w_i \, x_{t-i+1}}{\sum_{i=1}^{n} w_i}
$$

- WMA*t* = Weighted Moving Average of Time *t*
- N = Window Period Number
- X*t*-*i*+1 = Series Time Value *t* - *i* + 1
- W*i* = Weight Atribute

Python Code:

```python
    wvma = (data * weighter).rolling(window=window).sum() / volume.rolling(window=window).sum()
```

### - Standard Deviation:
$$
STD_i = \sqrt{ \frac{1}{n - 1} \sum_{i=0}^{n-1} (x_{t-i} - y_t)^2 }
$$

- STD*t* = Standard Deviation of Time *t*
- N = Window Period Number
- X*t*-*i* = Series of Time Value *t* - *i* 
- Y*t = Moving Average for for last N Values

Python Code:

```python
    std = data.rolling(window=window).std()
```

### - Weighted  Standard Deviation:
$$
WSTD_t = \sqrt{ \frac{ \sum_{i=0}^{n-1} w_i \, (x_{t-i} - y_{w,t})^2 }{ \sum_{i=0}^{n-1} w_i } }
$$

- WSTD*t* = Weighted Standard Deviation of Time *t*
- N = Window Period Number
- X*t*-*i* = Series of Time Value *t* - *i* 
- W*i* = Weight Atribute
- Y*t = Weighted Moving Average for for last N Values

Python Code:

```python
    wstd =  ((data** 2 * weighter).rolling(window=window).sum() / weighter.rolling(window=window).sum()) - (data.rolling(window=window).mean() ** 2) ** 0.5
```

### - Z-Score

$$
Z = \frac{x - ma}{std}
$$

- Z = Z-Score
- N = Value
- MA = Moving Average
- STD = Standard Deviation

Python Code:

```python
    z_score = (data - ma) / std 
    weigthed_z_score = (data - wma) / wstd 
```

### - Correlation

$$
CORR_t = \frac{
\sum_{i=0}^{n-1} (x_{t-i} - \bar{x}_t)(y_{t-i} - \bar{y}_t)
}{
\sqrt{ \sum_{i=0}^{n-1} (x_{t-i} - \bar{x}_t)^2 } \;
\sqrt{ \sum_{i=0}^{n-1} (y_{t-i} - \bar{y}_t)^2 }
}
$$


Python Code:

```python
    z_score = (data - ma) / std 
    weigthed_z_score = (data - wma) / wstd 
```

### 1 - Import Libraries

In [1]:
import pandas as pd
from itertools import combinations

### 2 - Import Data from Bitcoin Futures csv file and check the dataframe

In [2]:
df = pd.read_csv('bitcoin_futures_raw_data/futures_raw_data.csv')

In [3]:
df.tail()

,datetime,open_price,high_price,low_price,close_price,volume,buy_volume,transactions,buy_transactions,long_short_ratio,...,open_open_interest,high_open_interest,low_open_interest,close_open_interest,volume_delta,cumulative_volume_delta,transaction_delta,cumulative_transaction_delta,liquidation_delta,cumulative_liquidation_delta
2219,2025-10-10,121579.4,122497.0,102648.4,112714.9,283074.793,132251.301,3255324,1602863.0,0.9040,...,96722.011,97084.623,72364.700,72439.474,-18572.191,-2.274482e+06,-49598.0,-2854672.0,749.450,215490.762
2220,2025-10-11,112715.0,113300.1,109501.0,110579.1,162986.706,73542.497,2041944,983670.0,1.1858,...,72434.743,76479.026,71421.781,74324.913,-15901.712,-2.290384e+06,-74604.0,-2929276.0,77.889,215568.651
2221,2025-10-12,110579.2,115730.3,109509.3,114894.4,214563.112,107056.890,2305725,1149944.0,1.4021,...,74331.358,76356.016,70503.009,75910.074,-449.332,-2.290833e+06,-5837.0,-2935113.0,24.561,215593.212
2222,2025-10-13,114894.5,115912.0,113600.5,115111.9,190121.618,92858.401,1858297,927903.0,1.5556,...,75909.323,77513.040,75673.319,76664.481,-4404.816,-2.295238e+06,-2491.0,-2937604.0,-2.659,215590.553
2223,2025-10-14,115112.0,115360.2,109969.2,110665.2,145146.149,70047.343,1233227,616776.0,1.3299,...,76664.703,79261.712,75710.569,78994.003,-5051.463,-2.300289e+06,325.0,-2937279.0,103.858,215694.411


In [4]:
df.columns

Index(['datetime', 'open_price', 'high_price', 'low_price', 'close_price',
       'volume', 'buy_volume', 'transactions', 'buy_transactions',
       'long_short_ratio', 'long_liquidation', 'short_liquidation',
       'open_predicted_funding_rate', 'high_predicted_funding_rate',
       'low_predicted_funding_rate', 'close_predicted_funding_rate',
       'open_funding_rate', 'high_funding_rate', 'low_funding_rate',
       'close_funding_rate', 'open_open_interest', 'high_open_interest',
       'low_open_interest', 'close_open_interest', 'volume_delta',
       'cumulative_volume_delta', 'transaction_delta',
       'cumulative_transaction_delta', 'liquidation_delta',
       'cumulative_liquidation_delta'],
      dtype='object')

### 3 - List and Rename Columns to Calculate

In [5]:
# create a list of data to calculate the indicators
to_calculate_col = [
    'datetime',
    'close_price',
    'volume',
    'volume_delta',
    'cumulative_volume_delta',
    'transactions',
    'transaction_delta',
    'cumulative_transaction_delta',
    'long_short_ratio',
    'short_liquidation',
    'long_liquidation',
    'liquidation_delta',
    'cumulative_liquidation_delta',
    'close_predicted_funding_rate',
    'close_funding_rate',
    'close_open_interest'
]

### 4 - Function to Calculate the Indicators

#### 4.1 General indicators

In [6]:
# create the function
def indicator_func(df, to_calculate, timeframe, weighter=False):

    # if to weight is not needed
    if not weighter:
        ma =  df[to_calculate].rolling(window=timeframe).mean() # moving average
        std = df[to_calculate].rolling(window=timeframe).std() # standad deviation

    # if needs to me weighted by something
    else:
        ma = (
            (df[to_calculate] * df[weighter]).rolling(window=timeframe).sum() / # moving average
            df[weighter].rolling(window=timeframe).sum()
        )
        std = (
            (((df[to_calculate]** 2) * df[weighter]).rolling(window=timeframe).sum() / # standad deviation
             df[weighter].rolling(window=timeframe).sum()) - (ma ** 2)) ** 0.5
        
    z_score = (df[to_calculate] - ma) / std # z-score
    
    # create a new dataframe with just the datetime    
    df_new = df[['datetime']].copy()

    # add the indicators to new columns     
    df_new[f'{to_calculate}_{timeframe}_ma'] = ma
    df_new[f'{to_calculate}_{timeframe}_std'] = std
    df_new[f'{to_calculate}_{timeframe}_z_score'] = z_score

    # return the result
    return df_new

#### 4.2 Correlation

In [7]:
# create the function for calculate correlation
def correlation_func(df, correlator_1, correlator_2, timeframe):

        correlation = df[correlator_1].rolling(window=timeframe).corr(df[correlator_2])
        return correlation   

#### 4.3 Bollinger Bands

In [8]:
def band_func(df, to_calculate, timeframe, band_range = [2,4] , interval = 1, weighter=False):
        # if to weight is not needed
    if not weighter:
        ma =  df[to_calculate].rolling(window=timeframe).mean() # moving average
        std = df[to_calculate].rolling(window=timeframe).std() # standad deviation

    # if needs to me weighted by something
    else:
        ma = (
            (df[to_calculate] * df[weighter]).rolling(window=timeframe).sum() / # moving average
            df[weighter].rolling(window=timeframe).sum()
        )
        std = (
            (((df[to_calculate]** 2) * df[weighter]).rolling(window=timeframe).sum() / # standad deviation
             df[weighter].rolling(window=timeframe).sum()) - (ma ** 2)) ** 0.5
        
    # create a new dataframe with just the datetime    
    df_new = df[['datetime']].copy()


    # calculate the band and insert into a new column
    for band in range(band_range[0], band_range[1] + 1, interval):
        df_new[f'{to_calculate}_{timeframe}_{band}_upper'] = ma + band * std
        df_new[f'{to_calculate}_{timeframe}_{band}_lower'] = ma - band * std
  
    # return the result
    return df_new    

### 5 - Loop to calculate the Indicators

#### 5.1 General Indicators

In [9]:
# list of timeframes
timeframes = [31, 127, 367]

# new dataframe to store the indicators
df_indicators = df[[
    'datetime',
    'volume_delta',
    'cumulative_volume_delta',
    'transaction_delta',
    'cumulative_transaction_delta',
    'liquidation_delta',
    'cumulative_liquidation_delta'    
]]

# loop to call the function for each indicator and each timeframe
for col in to_calculate_col:
    if col != 'datetime':
        for timeframe in timeframes:
            if 'price' in col or 'interest' in col:
                df_indicators =  df_indicators.merge(indicator_func(df, col, timeframe, weighter='volume'), how='left', on='datetime')
            else:
                df_indicators =  df_indicators.merge(indicator_func(df, col, timeframe), how='left', on='datetime')

In [10]:
df_indicators.tail()

,datetime,volume_delta,cumulative_volume_delta,transaction_delta,cumulative_transaction_delta,liquidation_delta,cumulative_liquidation_delta,close_price_31_ma,close_price_31_std,close_price_31_z_score,...,close_funding_rate_367_z_score,close_open_interest_31_ma,close_open_interest_31_std,close_open_interest_31_z_score,close_open_interest_127_ma,close_open_interest_127_std,close_open_interest_127_z_score,close_open_interest_367_ma,close_open_interest_367_std,close_open_interest_367_z_score
2219,2025-10-10,-18572.191,-2.274482e+06,-49598.0,-2854672.0,749.450,215490.762,116611.070514,4428.226302,-0.879849,...,-0.973929,89858.294324,6421.129621,-2.712735,86276.423980,6169.549646,-2.242781,83442.420929,6866.780869,-1.602344
2220,2025-10-11,-15901.712,-2.290384e+06,-74604.0,-2929276.0,77.889,215568.651,116452.308994,4559.616660,-1.288093,...,-1.066086,89027.096852,7090.538736,-2.073493,86175.659856,6281.817291,-1.886516,83410.032828,6880.930403,-1.320333
2221,2025-10-12,-449.332,-2.290833e+06,-5837.0,-2935113.0,24.561,215593.212,116393.415273,4505.875918,-0.332680,...,-3.044628,88194.415852,7577.133795,-1.621239,86057.243226,6359.775734,-1.595523,83377.446776,6894.313085,-1.083121
2222,2025-10-13,-4404.816,-2.295238e+06,-2491.0,-2937604.0,-2.659,215590.553,116343.710358,4472.680932,-0.275408,...,-1.526989,87527.495562,7874.953301,-1.379439,85966.333212,6411.490221,-1.450810,83343.051253,6902.122007,-0.967611
2223,2025-10-14,-5051.463,-2.300289e+06,325.0,-2937279.0,103.858,215694.411,116144.081289,4546.242291,-1.205145,...,-0.999549,87195.903365,7941.851215,-1.032744,85900.517303,6453.044724,-1.070272,83324.949342,6898.961697,-0.627768


#### 5.2 Correlation

In [ ]:
# new dataframe to store the correlation
df_correlation =  df[['datetime']]
# for each indicator and price calculate the correlation between them in a certain timeframe
for timeframe in timeframes:
    for col1, col2 in combinations(to_calculate_col, 2):
        if col1 != 'datetime' and col2 != 'datetime':
            df_correlation[f'{col1}_{col2}_{timeframe}'] = correlation_func(df, col1, col2, timeframe)

In [12]:
df_correlation.tail()

,datetime,close_price_volume_31,close_price_volume_delta_31,close_price_cumulative_volume_delta_31,close_price_transactions_31,close_price_transaction_delta_31,close_price_cumulative_transaction_delta_31,close_price_long_short_ratio_31,close_price_short_liquidation_31,close_price_long_liquidation_31,...,liquidation_delta_cumulative_liquidation_delta_367,liquidation_delta_close_predicted_funding_rate_367,liquidation_delta_close_funding_rate_367,liquidation_delta_close_open_interest_367,cumulative_liquidation_delta_close_predicted_funding_rate_367,cumulative_liquidation_delta_close_funding_rate_367,cumulative_liquidation_delta_close_open_interest_367,close_predicted_funding_rate_close_funding_rate_367,close_predicted_funding_rate_close_open_interest_367,close_funding_rate_close_open_interest_367
2219,2025-10-10,0.177578,0.251219,0.610187,0.197256,0.103608,0.099319,-0.818695,0.270226,-0.153341,...,0.080024,-0.123602,-0.083450,-0.124294,-0.381150,-0.311649,-0.102422,0.589243,0.380901,0.401069
2220,2025-10-11,0.145323,0.322753,0.660946,0.114488,0.243428,0.257291,-0.823193,0.259287,-0.173315,...,0.080658,-0.124257,-0.084851,-0.126140,-0.384034,-0.315618,-0.109076,0.591702,0.385524,0.403332
2221,2025-10-12,0.120640,0.324106,0.632529,0.087432,0.247388,0.231419,-0.820570,0.255443,-0.177471,...,0.076224,-0.122029,-0.088236,-0.128285,-0.387200,-0.326069,-0.114651,0.596950,0.389340,0.407096
2222,2025-10-13,0.108686,0.328450,0.602658,0.076855,0.245925,0.218117,-0.810505,0.274551,-0.176183,...,0.077790,-0.122025,-0.090033,-0.127132,-0.391235,-0.328450,-0.120359,0.599916,0.392639,0.412156
2223,2025-10-14,0.094554,0.338703,0.639307,0.072858,0.228382,0.290606,-0.811369,0.292381,-0.182211,...,0.080893,-0.124981,-0.092819,-0.127557,-0.394515,-0.328924,-0.125455,0.600318,0.396840,0.417892


#### 5.3 Bollinger Bands

In [13]:
df_bands = band_func(df, 'close_price', 31, band_range = [2,4] , interval = 1, weighter='volume')

In [14]:
df_bands.tail()

,datetime,close_price_31_2_upper,close_price_31_2_lower,close_price_31_3_upper,close_price_31_3_lower,close_price_31_4_upper,close_price_31_4_lower
2219,2025-10-10,125467.523119,107754.617910,129895.749421,103326.391608,134323.975723,98898.165306
2220,2025-10-11,125571.542313,107333.075674,130131.158972,102773.459015,134690.775632,98213.842355
2221,2025-10-12,125405.167110,107381.663437,129911.043029,102875.787518,134416.918947,98369.911600
2222,2025-10-13,125289.072223,107398.348493,129761.753155,102925.667561,134234.434088,98452.986628
2223,2025-10-14,125236.565871,107051.596708,129782.808161,102505.354417,134329.050452,97959.112126


### 6 - Save as CSV file

In [15]:
# saves in CSV file    
#df_indicators.to_csv("indicators.csv", index=False) 